In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [6]:
def runSparsity (params: dict, diff_const = 1e-2):
    first_time = True

    for algorithm in params['algorithms']:
        for v_alpha in params['alphas']:
            for v_lamb in params['lambdas']:
                for seed in params['seeds']:

                    file_path = f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl"
                    data_tmp = pd.read_pickle(file_path)
                    if params['include_mask'] and "L1PSD" in params["algorithms"] and algorithm != "L1PSD":
                        data_l1psd = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_L1PSD_{v_lamb}_{v_alpha}_{seed}.pkl")
                        mask = data_l1psd["i"].to_numpy()
                        data_tmp = data_tmp.iloc[mask]

                    if first_time:
                        data = data_tmp.copy(deep=True)
                        first_time = False
                    else:
                        data = pd.concat((data, data_tmp), ignore_index=True)

                    print(f"Read {file_path}")

    data['diff'] = np.abs(data['x_r'] - data['x_0'])
    data['diff_count'] = data['diff'].apply(lambda x: np.sum(x > diff_const))
    return data

In [280]:
params = {}
# 'lr', 'nn'
params['base_model'] = 'lr'
# 'synthetic', 'german', 'sba'
params['data'] = 'german'
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
params['algorithms'] = ['Alg1', 'L1PSD']
params['include_mask'] = True

params['alphas']= [0.1]
# german_lr
params['lambdas'] = [0.5,0.3,0.1,0.04,0.01,0.004,0.001]
# german_nn
# params['lambdas'] = [0.7, 0.3, 0.1, 0.05, 0.01, 0.001]
# sba_lr
# params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]
# sba_nn
# params['lambdas'] = [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5]

sparsity_lambda = deepcopy(params['lambdas'])
sparsity_lambda.sort()
diff_const = 1e-2
df_results = runSparsity(params, diff_const=diff_const)

Read ../results/recourse/lr_german_Alg1_0.5_0.1_0.pkl
Read ../results/recourse/lr_german_Alg1_0.5_0.1_1.pkl
Read ../results/recourse/lr_german_Alg1_0.5_0.1_2.pkl
Read ../results/recourse/lr_german_Alg1_0.5_0.1_3.pkl
Read ../results/recourse/lr_german_Alg1_0.5_0.1_4.pkl
Read ../results/recourse/lr_german_Alg1_0.3_0.1_0.pkl
Read ../results/recourse/lr_german_Alg1_0.3_0.1_1.pkl
Read ../results/recourse/lr_german_Alg1_0.3_0.1_2.pkl
Read ../results/recourse/lr_german_Alg1_0.3_0.1_3.pkl
Read ../results/recourse/lr_german_Alg1_0.3_0.1_4.pkl
Read ../results/recourse/lr_german_Alg1_0.1_0.1_0.pkl
Read ../results/recourse/lr_german_Alg1_0.1_0.1_1.pkl
Read ../results/recourse/lr_german_Alg1_0.1_0.1_2.pkl
Read ../results/recourse/lr_german_Alg1_0.1_0.1_3.pkl
Read ../results/recourse/lr_german_Alg1_0.1_0.1_4.pkl
Read ../results/recourse/lr_german_Alg1_0.04_0.1_0.pkl
Read ../results/recourse/lr_german_Alg1_0.04_0.1_1.pkl
Read ../results/recourse/lr_german_Alg1_0.04_0.1_2.pkl
Read ../results/recourse/

In [281]:
df_results['algorithm'] = df_results['algorithm'].replace('alg1',"Alg1")
df_results['algorithm'] = df_results['algorithm'].replace('ROAR',"ROARLInf")

df_results_mean = df_results.groupby(['algorithm', 'alpha', 'lambda'], as_index=False).mean()

In [282]:
# df_results_mean = df_results_mean.sort_values(['alpha', 'lambda', 'algorithm'])
# df_results_mean_sam = df_results_mean.iloc[0:len(params['algorithms'])]
# if "ROARLInf" in params['algorithms'] and "ROARL1" in params['algorithms']:
#     df_results_mean_sam.iloc[[2,3]] = df_results_mean_sam.iloc[[3,2]]
# stacked  = np.stack(df_results_mean_sam['diff'].apply(lambda x: np.where(x > diff_const, x, 0)))

# fig = px.imshow(stacked.round(1), 
#             color_continuous_scale="Reds",
#             y=df_results_mean_sam['algorithm'].to_list(),
#             text_auto=True,
#             labels=dict(x='Features', y='Algorithms', color='Avg Cost'),
#             title=f"German LR Alpha={df_results_mean_sam['alpha'].unique()} Lambda={df_results_mean_sam['lambda'].unique()}")

# fig.show()

In [283]:
df_results_mean_histo = df_results_mean.copy(deep=True)
df_results_mean_histo['lambda_str'] = df_results_mean_histo['lambda'].astype(str)

df_results_mean = df_results_mean.sort_values(['alpha', 'lambda', 'algorithm'], ascending=[True, True, False])
df_results_mean

,algorithm,alpha,lambda,seed,i,x_0,x_r,theta_0,diff,diff_count
7,L1PSD,0.1,0.001,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-6.69367794117647, 1.8353279411764707, -0.328...","[-0.42170294117647034, -0.09014558823529416, 0...","[8.830848529411767, 2.941176470598524e-06, 1.4...",2.544118
0,Alg1,0.1,0.001,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-14.145097058823534, 1.835330882352941, -0.26...","[-0.42170294117647034, -0.09014558823529416, 0...","[16.282267647058823, 0.0, 0.0670264705882353, ...",1.632353
8,L1PSD,0.1,0.004,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-5.284941176470587, 1.835330882352941, -0.328...","[-0.42170294117647034, -0.09014558823529416, 0...","[7.42211176470588, 0.0, 0.0, 2.757939705882353...",2.382353
1,Alg1,0.1,0.004,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-10.514317647058824, 1.835330882352941, -0.26...","[-0.42170294117647034, -0.09014558823529416, 0...","[12.651488235294119, 0.0, 0.0670264705882353, ...",1.632353
9,L1PSD,0.1,0.010,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-4.294583823529414, 1.835330882352941, -0.328...","[-0.42170294117647034, -0.09014558823529416, 0...","[6.4317544117647065, 0.0, 2.9411764705879115e-...",2.000000
2,Alg1,0.1,0.010,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-8.076942647058823, 1.835330882352941, -0.261...","[-0.42170294117647034, -0.09014558823529416, 0...","[10.214113235294118, 0.0, 0.0670264705882353, ...",1.632353
10,L1PSD,0.1,0.040,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-2.4059161764705874, 1.835330882352941, -0.32...","[-0.42170294117647034, -0.09014558823529416, 0...","[4.543086764705882, 0.0, 5.882352941172558e-06...",1.970588
3,Alg1,0.1,0.040,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-4.174604411764706, 1.835330882352941, -0.261...","[-0.42170294117647034, -0.09014558823529416, 0...","[6.311774999999997, 0.0, 0.0670264705882353, 1...",1.632353
11,L1PSD,0.1,0.100,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-1.2117911764705886, 1.8353264705882353, -0.3...","[-0.42170294117647034, -0.09014558823529416, 0...","[3.348961764705882, 4.411764705875337e-06, 7.3...",1.485294
4,Alg1,0.1,0.100,1.985294,6.426471,"[2.1371705882352927, 1.835330882352941, -0.328...","[-1.238833823529412, 1.835330882352941, -0.263...","[-0.42170294117647034, -0.09014558823529416, 0...","[3.3760044117647054, 0.0, 0.06519117647058824,...",1.617647


In [284]:
fig = px.histogram(df_results_mean_histo, 
                   x = "lambda_str",
                   y = "diff_count",
                   color="algorithm",
                   barmode="group",
                   title=f"{params['data']}_{params['base_model']}_alpha=0.1_histogram")
fig.show()

In [285]:
from plotly.subplots import make_subplots

width_oneF = 150
height_oneF = 1000 / len(params["algorithms"])

fig_sub = make_subplots(rows = len(sparsity_lambda), 
                        cols = 1, 
                        shared_xaxes=True,
                        subplot_titles=[f"Lambda {val}" for val in sparsity_lambda],
                        x_title="Features",
                        y_title="Algorithms")

for i,lamb in enumerate(sparsity_lambda):
    df_results_mean_tmp = df_results_mean[df_results_mean['lambda'] == lamb]
    if "ROARLInf" in params['algorithms'] and "ROARL1" in params['algorithms']:
        df_results_mean_tmp.iloc[[0,1]] = df_results_mean_tmp.iloc[[1,0]]
    stacked  = np.stack(df_results_mean_tmp['diff'])

    fig_sub.add_trace(go.Heatmap(z=stacked.round(2),
                                x=np.arange(stacked.shape[1]),
                                y=df_results_mean_tmp['algorithm'].to_list(),
                                coloraxis="coloraxis",
                                texttemplate="%{z}"), 
                                row=i+1, col=1)

fig_sub.update_xaxes(tickmode="array", 
                     tickvals=np.arange(stacked.shape[1]), 
                     row=len(sparsity_lambda), 
                     col=1)
# fig_sub.update_yaxes(title_text="Algorithms", row=len(sparsity_lambda) // 2, col=1)
fig_sub.update_layout(
    coloraxis=dict(colorscale="Reds"),
    coloraxis_colorbar=dict(
        title="Avg. Cost",
    ),
    width=width_oneF * stacked.shape[1],
    height=height_oneF * stacked.shape[0],
    title_text = f"{params['data']}_{params['base_model']}_alpha=0.1_Sparsity"
)

In [ ]:
# figNameHisto = f"{params['base_model']}_{params['data']}_histogram.html" 
# figNameSparsity = f"{params['base_model']}_{params['data']}_sparsity.html"
# fig.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-11\\" + 
#       figNameHisto + f".html")
# fig_sub.write_html(f"C:\\Users\\pmyat\\OneDrive - Drexel University\\ResearchAssistantCoop_Jabbari\\" +
#       f"experimentResults\\meetings\\2025-09-11\\" + 
#       figNameSparsity + f".html")

In [278]:
df_results[(df_results['seed'] == 0) & (df_results['i'] == 24) & (df_results['lambda'] == 0.001)]

,algorithm,seed,alpha,lambda,i,x_0,x_r,theta_0,diff,diff_count
175,Alg1,0,0.1,0.001,24,"[1.2526, -0.3361, -1.0155, 0.0, 0.0, 0.0, 1.0]","[1.2526, 5.1217, -1.0155, 0.0, 0.0, 0.0, 1.0]","[-1.3217, 1.6803, 0.7039, -0.1972, -0.0802, 0....","[0.0, 5.4578, 0.0, 0.0, 0.0, 0.0, 0.0]",1
385,L1PSD,0,0.1,0.001,24,"[1.2526, -0.3361, -1.0155, 0.0, 0.0, 0.0, 1.0]","[-0.728, 2.7053, -1.0155, 0.0, 0.0, -0.0, 1.0]","[-1.4307, 1.8215, 0.7708, -0.1706, -0.1865, 0....","[1.9806, 3.0414, 0.0, 0.0, 0.0, 0.0, 0.0]",2
